# FrameCheck — CIFAKE Detector: Evaluation Notebook

**Purpose:** Recompute metrics from `cifake_detector.keras` on the CIFAKE test split.  
No training occurs in this notebook. Metrics are recomputed without fitting, but this split was used as validation during training and is not a fresh unseen test set.

**Scope:** Results labelled *CIFAKE test split* apply only to 32×32 CIFAKE-format images.  
A separate section (Section 7) handles non-CIFAKE images if you supply them.

**Data leakage note:** The original training notebook used the CIFAKE test split as `validation_data` during `model.fit`. That means the training-time accuracy (90.85%) was computed on this same split epoch-by-epoch, and early stopping (if applied) would have been guided by it. This notebook loads the saved weights and makes one prediction pass — no fitting, fine-tuning, or threshold selection. Its metrics are a re-evaluation of the validation split, not independent evidence on unseen data.

---

## How to run

### Option A — Local (VS Code / Jupyter)
```
# From the project root
pip install tensorflow-cpu pillow numpy scikit-learn matplotlib seaborn tqdm
jupyter notebook notebooks/evaluate_cifake.ipynb
```
The notebook auto-detects the model at `models/cifake_detector.keras` and the test set at `imagesdataset/test/`.

### Option B — Google Colab
1. Upload this notebook to Colab.
2. Upload `cifake_detector.keras` using the Colab file panel (or mount Drive).
3. Upload the CIFAKE test set as a zip named `test.zip` with structure `test/FAKE/` and `test/REAL/`.
4. Set `RUNNING_IN_COLAB = True` in the **Configuration** cell below.
5. Run all cells.

## 0. Install dependencies (Colab only)

In [ ]:
# Only runs when RUNNING_IN_COLAB is True (set in Section 1 below).
# This cell is intentionally placed before imports so Colab has packages ready.
import sys, os
RUNNING_IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if RUNNING_IN_COLAB:
    !pip install -q tensorflow-cpu pillow numpy scikit-learn matplotlib seaborn tqdm
    print('Packages installed.')
else:
    print('Local environment detected — skipping pip install.')

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# ── Detect environment ──────────────────────────────────────────────────────
import sys
RUNNING_IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

# ── Paths ───────────────────────────────────────────────────────────────────
if RUNNING_IN_COLAB:
    # When running in Colab, upload cifake_detector.keras to /content/
    # and upload your test split zip as /content/test.zip
    # (structure inside zip: test/FAKE/*.jpg and test/REAL/*.jpg)
    MODEL_PATH   = Path('/content/cifake_detector.keras')
    TEST_DIR     = Path('/content/test')         # after unzipping test.zip
    CUSTOM_DIR   = Path('/content/custom_test')  # optional: non-CIFAKE images

    # Unzip test set if present
    if Path('/content/test.zip').exists() and not TEST_DIR.exists():
        import zipfile
        with zipfile.ZipFile('/content/test.zip') as z:
            z.extractall('/content/')
        print('Unzipped test set.')
else:
    # Local paths — adjust if your layout differs
    PROJECT_ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path('.').resolve()
    # When running from notebooks/ directory, go up one level
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    MODEL_PATH = PROJECT_ROOT / 'models' / 'cifake_detector.keras'
    TEST_DIR   = PROJECT_ROOT / 'imagesdataset' / 'test'
    CUSTOM_DIR = PROJECT_ROOT / 'custom_test'   # optional; create if needed

# ── Evaluation settings ─────────────────────────────────────────────────────
IMAGE_SIZE   = (32, 32)    # must match training preprocessing
BATCH_SIZE   = 256         # larger batches = faster inference; reduce if OOM
THRESHOLD    = 0.5         # sigmoid threshold: >= THRESHOLD → REAL
RANDOM_SEED  = 42

# ── Sanity checks ───────────────────────────────────────────────────────────
print(f'Model path : {MODEL_PATH}')
print(f'Model found: {MODEL_PATH.is_file()}')
print(f'Test dir   : {TEST_DIR}')
print(f'Test found : {TEST_DIR.is_dir()}')
if not MODEL_PATH.is_file():
    print()
    print('⚠  MODEL FILE NOT FOUND.')
    print('   Export cifake_detector.keras from Colab with:')
    print('   from google.colab import files')
    print("   files.download('/content/cifake_detector.keras')")
    print('   Then place the file at the path shown above.')
if not TEST_DIR.is_dir():
    print()
    print('⚠  TEST DIRECTORY NOT FOUND.')
    print('   In Colab: upload test.zip (structure: test/FAKE/*.jpg, test/REAL/*.jpg).')
    print('   Locally:  confirm imagesdataset/test/ exists and contains FAKE/ and REAL/.')

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # suppress TF info/warning logs

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, ImageOps
from pathlib import Path
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
)
from sklearn.calibration import calibration_curve
import tensorflow as tf
from tensorflow import keras

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

## 3. Load the model

In [ ]:
assert MODEL_PATH.is_file(), (
    f'Model file not found at {MODEL_PATH}.\n'
    'Cannot proceed without it. See configuration cell for export instructions.'
)

model = keras.models.load_model(MODEL_PATH, compile=False)
print('Model loaded successfully.')
print()
model.summary()
print()
print('Input  shape:', model.input_shape)
print('Output shape:', model.output_shape)

## 4. Load and preprocess the CIFAKE test set

Preprocessing **exactly mirrors** `backend/app.py`:
- Load image, EXIF-transpose, convert to RGB
- Resize to 32 × 32 (bilinear)
- Cast to `float32` — **no division by 255** (the model's `Rescaling` layer handles that internally)

**Label assignment:** folder name `REAL` → label `1`, `FAKE` → label `0`.  
This matches the alphabetical ordering used by `image_dataset_from_directory` during training.

In [ ]:
assert TEST_DIR.is_dir(), f'Test directory not found: {TEST_DIR}'

# Class → integer label mapping.
# image_dataset_from_directory sorts alphabetically: FAKE=0, REAL=1.
CLASS_TO_LABEL = {'FAKE': 0, 'REAL': 1}
LABEL_TO_CLASS = {0: 'FAKE', 1: 'REAL'}

def load_image(path: Path) -> np.ndarray:
    """Load one image and preprocess it exactly as backend/app.py does."""
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img).convert('RGB')
        img = img.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
    # Raw [0, 255] float32 — model's Rescaling layer divides by 255 internally.
    return np.asarray(img, dtype=np.float32)

def load_split(root: Path, desc: str) -> tuple[np.ndarray, np.ndarray, list[Path]]:
    """Load all images in root/FAKE/ and root/REAL/."""
    images, labels, paths = [], [], []
    for class_name, label_int in CLASS_TO_LABEL.items():
        class_dir = root / class_name
        if not class_dir.is_dir():
            print(f'  WARNING: {class_dir} not found — skipping.')
            continue
        files = sorted(class_dir.iterdir())
        print(f'  {class_name}: {len(files)} images')
        for fp in tqdm(files, desc=f'  Loading {class_name}', leave=False):
            if fp.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp', '.gif'}:
                continue
            try:
                images.append(load_image(fp))
                labels.append(label_int)
                paths.append(fp)
            except Exception as e:
                print(f'  Skipped {fp.name}: {e}')
    return np.stack(images), np.array(labels, dtype=np.int32), paths

print(f'Loading CIFAKE test set from: {TEST_DIR}')
X_test, y_test, test_paths = load_split(TEST_DIR, 'CIFAKE test')
print(f'\nTotal loaded: {len(X_test)} images')
print(f'Label distribution: FAKE={np.sum(y_test==0)}, REAL={np.sum(y_test==1)}')
print(f'X_test shape: {X_test.shape}, dtype: {X_test.dtype}')
print(f'Pixel range: [{X_test.min():.0f}, {X_test.max():.0f}] (expected [0, 255])')

### 4a. Visual sanity check — sample images

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for row, cls in enumerate(['FAKE', 'REAL']):
    idx_cls = np.where(y_test == CLASS_TO_LABEL[cls])[0]
    sample_idx = rng.choice(idx_cls, size=8, replace=False)
    for col, i in enumerate(sample_idx):
        axes[row, col].imshow(X_test[i].astype(np.uint8))
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=9, fontweight='bold', loc='left', pad=3)
plt.suptitle('CIFAKE test set — random sample (FAKE top, REAL bottom)', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Run inference — CIFAKE test split

Single forward pass. No fitting, no fine-tuning.

In [ ]:
print(f'Running inference on {len(X_test)} images (batch size={BATCH_SIZE})…')
y_prob_real = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1).reshape(-1)
# y_prob_real[i] = P(REAL) for image i
y_prob_fake = 1.0 - y_prob_real
y_pred = (y_prob_real >= THRESHOLD).astype(np.int32)  # 1=REAL, 0=FAKE

print()
print(f'Sigmoid output range : [{y_prob_real.min():.4f}, {y_prob_real.max():.4f}]')
print(f'Predicted REAL       : {np.sum(y_pred == 1)}')
print(f'Predicted FAKE       : {np.sum(y_pred == 0)}')

## 6. Metrics — CIFAKE test split (recomputed post-training)

> All results below apply **only to 32×32 CIFAKE-format images**.
> Do not generalise these figures to full-resolution images or other generators.

### 6a. Scalar metrics

In [ ]:
acc    = accuracy_score(y_test, y_pred)
prec   = precision_score(y_test, y_pred, pos_label=1)   # REAL as positive
rec    = recall_score(y_test, y_pred, pos_label=1)
f1     = f1_score(y_test, y_pred, pos_label=1)
prec_f = precision_score(y_test, y_pred, pos_label=0)   # FAKE as positive
rec_f  = recall_score(y_test, y_pred, pos_label=0)
f1_f   = f1_score(y_test, y_pred, pos_label=0)
mac_f1 = f1_score(y_test, y_pred, average='macro')
wgt_f1 = f1_score(y_test, y_pred, average='weighted')
roc    = roc_auc_score(y_test, y_prob_real)

print('=' * 62)
print(' CIFAKE TEST SPLIT — Recomputed post-training evaluation')
print(' (single forward pass; no training in this notebook)')
print('=' * 62)
print(f'  Images evaluated  : {len(X_test):,}')
print(f'  Decision threshold: {THRESHOLD}')
print()
print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print()
print(f'  --- Class REAL (positive) ---')
print(f'  Precision         : {prec:.4f}')
print(f'  Recall            : {rec:.4f}')
print(f'  F1                : {f1:.4f}')
print()
print(f'  --- Class FAKE (positive) ---')
print(f'  Precision         : {prec_f:.4f}')
print(f'  Recall            : {rec_f:.4f}')
print(f'  F1                : {f1_f:.4f}')
print()
print(f'  Macro F1          : {mac_f1:.4f}')
print(f'  Weighted F1       : {wgt_f1:.4f}')
print(f'  ROC-AUC           : {roc:.4f}')
print('=' * 62)
print()

# Compare with training-time reported accuracy
reported = 0.9085
delta = acc - reported
print(f'  Training-time reported accuracy : {reported:.4f} (90.85%)')
print(f'  This evaluation accuracy        : {acc:.4f} ({acc*100:.2f}%)')
print(f'  Difference                      : {delta:+.4f}')
if abs(delta) < 0.005:
    print('  ✓ Results are consistent with training-time report.')
elif delta < 0:
    print('  ⚠  Slightly lower than training-time report — consistent with')
    print('     test-as-validation effect (expected).')
else:
    print('  ⚠  Unexpected difference — verify preprocessing matches training.')

### 6b. Classification report

In [ ]:
print('Classification report — CIFAKE test split')
print(classification_report(y_test, y_pred,
                             target_names=['FAKE (0)', 'REAL (1)'],
                             digits=4))

### 6c. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

class_names = ['FAKE', 'REAL']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2%'],
    ['Confusion matrix — counts', 'Confusion matrix — row-normalised'],
):
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.5, linecolor='#ccc',
        annot_kws={'size': 13},
    )
    ax.set_xlabel('Predicted label', fontsize=11)
    ax.set_ylabel('True label', fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)

plt.suptitle(
    'CIFAKE test split — recomputed evaluation',
    fontsize=11, y=1.01,
)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN (FAKE→FAKE): {tn:,}   FP (FAKE→REAL): {fp:,}')
print(f'FN (REAL→FAKE): {fn:,}   TP (REAL→REAL): {tp:,}')

### 6d. ROC curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob_real)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'ROC (AUC = {roc:.4f})')
ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random')
# Mark the operating point at THRESHOLD
op_idx = np.argmin(np.abs(thresholds - THRESHOLD))
ax.scatter(fpr[op_idx], tpr[op_idx], color='tomato', s=80, zorder=5,
           label=f'Threshold = {THRESHOLD}')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — CIFAKE test split', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.01])
plt.tight_layout()
plt.show()

### 6e. Calibration plot (reliability diagram)

Shows whether the model's P(REAL) outputs are well-calibrated.  
A perfectly calibrated model follows the diagonal.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_prob_real, n_bins=20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Reliability diagram
axes[0].plot(prob_pred, prob_true, 's-', color='steelblue', label='Model')
axes[0].plot([0, 1], [0, 1], '--', color='grey', label='Perfect calibration')
axes[0].set_xlabel('Mean predicted P(REAL)', fontsize=11)
axes[0].set_ylabel('Fraction of REAL images', fontsize=11)
axes[0].set_title('Reliability diagram (calibration)', fontsize=12)
axes[0].legend(fontsize=10)

# Predicted probability histogram
axes[1].hist(y_prob_real[y_test == 0], bins=50, alpha=0.6,
             color='tomato', label='FAKE (true)')
axes[1].hist(y_prob_real[y_test == 1], bins=50, alpha=0.6,
             color='steelblue', label='REAL (true)')
axes[1].axvline(THRESHOLD, color='black', linestyle='--', linewidth=1.2,
                label=f'Threshold = {THRESHOLD}')
axes[1].set_xlabel('P(REAL) predicted by model', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Score distribution by true class', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle('CIFAKE test split — calibration analysis', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

### 6f. Example predictions

Top row: **correct** predictions. Bottom row: **incorrect** (model mistakes).  
Confidence shown is the model's predicted probability for the assigned class.

In [ ]:
def show_examples(X, y_true, y_pred_arr, y_prob_arr, n_each=8, seed=RANDOM_SEED):
    """Show n_each correct and n_each incorrect predictions side by side."""
    rng = np.random.default_rng(seed)

    correct_idx   = np.where(y_pred_arr == y_true)[0]
    incorrect_idx = np.where(y_pred_arr != y_true)[0]

    n_correct   = min(n_each, len(correct_idx))
    n_incorrect = min(n_each, len(incorrect_idx))

    c_sample = rng.choice(correct_idx,   size=n_correct,   replace=False)
    e_sample = rng.choice(incorrect_idx, size=n_incorrect, replace=False)

    print(f'Showing {n_correct} correct and {n_incorrect} incorrect examples.')
    if len(incorrect_idx) == 0:
        print('(No incorrect predictions — model got everything right!)')

    fig, axes = plt.subplots(2, max(n_correct, n_incorrect),
                              figsize=(max(n_correct, n_incorrect) * 1.8, 4.5))
    row_labels = [f'Correct ({n_correct})', f'Incorrect ({n_incorrect})']

    for row, (sample, label) in enumerate([
        (c_sample, '✓ Correct'),
        (e_sample, '✗ Incorrect'),
    ]):
        for col in range(max(n_correct, n_incorrect)):
            ax = axes[row, col]
            if col < len(sample):
                i = sample[col]
                prob_real  = y_prob_arr[i]
                prob_fake  = 1.0 - prob_real
                pred_cls   = LABEL_TO_CLASS[y_pred_arr[i]]
                true_cls   = LABEL_TO_CLASS[y_true[i]]
                conf       = prob_real if y_pred_arr[i] == 1 else prob_fake
                color      = 'green' if y_pred_arr[i] == y_true[i] else 'red'
                ax.imshow(X[i].astype(np.uint8))
                ax.set_title(
                    f'Pred: {pred_cls}\nTrue: {true_cls}\nConf: {conf:.2f}',
                    fontsize=7, color=color, pad=3
                )
            ax.axis('off')
        axes[row, 0].set_ylabel(row_labels[row], fontsize=9, rotation=90,
                                 labelpad=4)

    plt.suptitle('Example predictions — CIFAKE test split\n'
                 '(green = correct, red = incorrect)',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

show_examples(X_test, y_test, y_pred, y_prob_real, n_each=8)

## 7. Non-CIFAKE evaluation (optional)

**Purpose:** Evaluate the model on images from sources other than CIFAKE — e.g., Midjourney,  
DALL-E, Firefly, or real photographs from the web.

**To use this section:**
1. Create a directory with this structure:
   ```
   custom_test/
   ├── FAKE/   ← AI-generated images (any resolution; will be resized to 32×32)
   └── REAL/   ← Genuine photographs
   ```
2. Locally: place it at `<project_root>/custom_test/`  
   In Colab: upload a `custom_test.zip` to `/content/`
3. Run this cell.

> **Results from this section are reported separately and must not be merged with CIFAKE figures.**
> The model was not trained on these images; generalisability is unknown.

In [ ]:
# Unzip custom_test.zip if present in Colab
if RUNNING_IN_COLAB and Path('/content/custom_test.zip').exists() and not CUSTOM_DIR.exists():
    import zipfile
    with zipfile.ZipFile('/content/custom_test.zip') as z:
        z.extractall('/content/')
    print('Unzipped custom_test.zip.')

if not CUSTOM_DIR.is_dir():
    print(f'Custom test directory not found at {CUSTOM_DIR}.')
    print('Skipping non-CIFAKE evaluation.')
    print('See cell instructions above for how to provide images.')
else:
    print(f'\nLoading custom test images from: {CUSTOM_DIR}')
    X_custom, y_custom, custom_paths = load_split(CUSTOM_DIR, 'custom')

    if len(X_custom) == 0:
        print('No images loaded from custom_test/. Check directory structure.')
    else:
        print(f'Total loaded: {len(X_custom)} images')
        y_prob_custom = model.predict(X_custom, batch_size=BATCH_SIZE, verbose=1).reshape(-1)
        y_pred_custom = (y_prob_custom >= THRESHOLD).astype(np.int32)

        # Metrics — labelled separately from CIFAKE
        print()
        print('=' * 62)
        print(' NON-CIFAKE CUSTOM TEST SET — separate evaluation')
        print(f' Source: {CUSTOM_DIR}')
        print(' NOTE: Model trained on CIFAKE only. These results may not')
        print(' reflect true performance on out-of-distribution images.')
        print('=' * 62)
        print(f'  Images evaluated  : {len(X_custom):,}')
        print(f'  FAKE : {np.sum(y_custom==0):,}   REAL : {np.sum(y_custom==1):,}')
        print()

        if len(np.unique(y_custom)) < 2:
            print('  Only one class present — ROC-AUC and some metrics undefined.')
        else:
            print(classification_report(y_custom, y_pred_custom,
                                         target_names=['FAKE (0)', 'REAL (1)'],
                                         digits=4))
            roc_custom = roc_auc_score(y_custom, y_prob_custom)
            print(f'  ROC-AUC: {roc_custom:.4f}')

            # Confusion matrix
            cm_c = confusion_matrix(y_custom, y_pred_custom)
            fig, ax = plt.subplots(figsize=(5, 4))
            sns.heatmap(cm_c, annot=True, fmt='d', cmap='Oranges',
                        xticklabels=['FAKE', 'REAL'],
                        yticklabels=['FAKE', 'REAL'],
                        ax=ax, linewidths=0.5)
            ax.set_xlabel('Predicted')
            ax.set_ylabel('True')
            ax.set_title('Non-CIFAKE custom test set\n(reported separately from CIFAKE results)')
            plt.tight_layout()
            plt.show()

            show_examples(X_custom, y_custom, y_pred_custom, y_prob_custom, n_each=6)

## 8. Summary and caveats

Run this cell last to print a final summary.

In [ ]:
print('=' * 62)
print(' EVALUATION SUMMARY')
print('=' * 62)
print()
print('Dataset  : CIFAKE (Bird & Lotfi, 2024)')
print(f'Test set : {len(X_test):,} images ({np.sum(y_test==1):,} REAL, {np.sum(y_test==0):,} FAKE)')
print(f'Model    : {MODEL_PATH.name}')
print(f'Threshold: {THRESHOLD}')
print()
print(f'  Accuracy (recomputed)  : {acc*100:.2f}%')
print(f'  Macro F1               : {mac_f1:.4f}')
print(f'  ROC-AUC                : {roc:.4f}')
print()
print('CAVEATS')
print('  1. These results apply ONLY to 32x32 CIFAKE-format images.')
print('  2. FAKE images are Stable Diffusion v1.4 only.')
print('     Generalisability to other generators is unknown.')
print('  3. The test split was used as validation_data during model.fit.')
print('     No fitting occurs here, but this split was used as validation')
print('     during training, so it is not an unseen independent test.')
print('  4. All outputs are ESTIMATES, not verdicts.')
print('=' * 62)